In [ ]:
!pip install torch torchvision open_clip_torch faiss-cpu Pillow numpy

In [ ]:
import torch
import open_clip
import faiss
import numpy as np
from PIL import Image
import os
import json
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("salargamer/coco-dataset")

print("Path to dataset files:", path)

In [ ]:
cp -r /root/.cache/kagglehub/datasets/salargamer/coco-dataset/versions/1 /backend/data/images/

In [ ]:
IMAGE_DIR = "/backend/data/images/coco_data/images/val2017"
INDEX_PATH = "faiss_index.bin"
MAPPING_PATH = "image_paths.json"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()

image_paths = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
image_mapping = {}
embeddings = []

print(f"Extracting features from {len(image_paths)} images...")

with torch.no_grad():
    for idx, img_path in enumerate(tqdm(image_paths)):
        try:
            # Load and preprocess image
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)

            # Generate embedding and normalize
            embedding = model.encode_image(img_tensor)
            embedding /= embedding.norm(dim=-1, keepdim=True)

            embeddings.append(embedding.cpu().numpy())
            image_mapping[idx] = os.path.basename(img_path) # Only store file name

        except Exception as e:
            print(f"Error at {img_path}: {e}")

embeddings = np.vstack(embeddings).astype('float32')

In [ ]:
# Build FAISS Index
d = embeddings.shape[1] # (512 for ViT-B-32)
index = faiss.IndexFlatIP(d)
index.add(embeddings)

In [ ]:
# Save
faiss.write_index(index, INDEX_PATH)
with open(MAPPING_PATH, 'w') as f:
    json.dump(image_mapping, f)

print(f"Index successfully created and saved. Please download {INDEX_PATH} and {MAPPING_PATH}.")